# Scaling & Hyperparameter Tuning

![Scaling and hyperparameter tuning illustration](https://i.ytimg.com/vi/b0L_2jKEbA4/maxresdefault.jpg)

In this notebook you will see a short example of how to scale your data and tune the hyperparameters of your models using grid or randomized search. Along the way you will use cross-validation to evaluate your models more reliably than a single train-test split allows.

Use the raw Titanic CSV included in this repository to compare feature scaling and hyperparameter searches. Imputation and encoding stay **inside each pipeline**, so validation rows do not influence preprocessing.

The older `titanic_preprocessed.csv` remains available for reference, but is not used here because its preprocessing was done before this notebook's split. We keep the same six source features (`Age`, `SibSp`, `Parch`, `Fare`, `Sex`, `Embarked`); the change is when preprocessing is fitted, not which information the models receive.

**Next notebook:** here, cross-validation selects the model within the training set. The heart exercise solution adds a separate validation set to compare candidates after tuning on training folds. Both protocols keep the test set for one final evaluation; a separate validation set is not always required.

## Learning Objectives

By the end of this notebook, you should be able to:

- Scale numerical features with standardization (`StandardScaler`) and min-max scaling (`MinMaxScaler`), and explain when each is useful.
- Combine preprocessing and a model into a single scikit-learn `Pipeline`.
- Evaluate models with K-fold cross-validation instead of relying on one train-test split.
- Tune hyperparameters with `GridSearchCV` and `RandomizedSearchCV`, and compare the trade-offs between them.

In [ ]:
from timeit import default_timer as timer

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import (
    GridSearchCV,
    RandomizedSearchCV,
    StratifiedKFold,
    cross_val_score,
    train_test_split,
)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Reuse one shuffled, stratified split for every model and both searches.
RSEED = 10
N_SPLITS = 5  # Keep the same number of validation folds for all comparisons.
cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RSEED)

## The Titanic Dataset

The bundled `data/titanic.csv` contains **891 labeled passenger records**: one row per passenger and 12 columns. We predict `Survived`, a binary outcome: **0 = did not survive; 1 = survived**. This is a historical classification exercise, not a system for deciding who should receive assistance.

The CSV follows the public [Titanic competition data dictionary](https://www.kaggle.com/competitions/titanic/data):

| Column | Meaning | Role in this notebook |
| --- | --- | --- |
| `PassengerId` | Passenger identifier | Excluded |
| `Survived` | Survival outcome, 0 or 1 | Target to predict |
| `Pclass` | Ticket class: 1st, 2nd or 3rd | Excluded from this bounded example |
| `Name` | Passenger name | Excluded |
| `Sex` | Recorded sex: female or male | Categorical input |
| `Age` | Age in years; fractional ages occur | Numerical input |
| `SibSp` | Number of siblings/spouses aboard | Numerical input |
| `Parch` | Number of parents/children aboard | Numerical input |
| `Ticket` | Ticket number | Excluded |
| `Fare` | Passenger fare | Numerical input |
| `Cabin` | Cabin number | Excluded |
| `Embarked` | Port: C = Cherbourg, Q = Queenstown, S = Southampton | Categorical input |

We create our **own train/test split** from these labeled records. The competition's separate unlabeled test file and a leaderboard submission are not needed.

The labels are imbalanced: 549 passengers did not survive and 342 survived. This is why we later compare accuracy with a majority-class baseline. First inspect a few rows and missing values. In this file, `Age` has 177 missing values, `Cabin` has 687, and `Embarked` has 2. Missing values are handled later inside the pipeline; we do not fill them using statistics from the full dataset.

In [ ]:
# Load raw data so preprocessing can be learned inside each training fold.
df = pd.read_csv("data/titanic.csv")
df.head()

In [ ]:
# Count missing values in each column before deciding how to handle them.
df.isnull().sum()

## Train-Test Split

`X` contains the **input features**; `y` contains the **target label** we want to predict. Each row in `X` and `y` refers to the same passenger.

### Why These Six Inputs?

We keep four numerical inputs (`Age`, `SibSp`, `Parch`, `Fare`) and two categorical inputs (`Sex`, `Embarked`). They provide a small mix of ages, counts, fares and categories to practice imputation, scaling and encoding. This is a **teaching choice**, not a claim that these are the best predictors.

Why not use every column?

- `Survived` is the answer. Including it in `X` would leak the target.
- `PassengerId` is an identifier, not a meaningful passenger measurement.
- `Name` and `Ticket` need feature engineering before they can be useful here; their raw text is not a ready-to-use numerical input.
- `Cabin` is mostly missing and needs additional decisions about representation and missingness.
- `Pclass` is a plausible predictor too. It is omitted to keep the same six-input example and the input contract reused in Notebook 3, **not because it is irrelevant**.

Adding features is a valid extension: update preprocessing and compare using training cross-validation, not the test score. If you export that model, update Notebook 3's input schema too.

In [ ]:
# Separate the label from the six inputs; never give Survived to the classifier.
y = df["Survived"]
X = df[["Age", "SibSp", "Parch", "Fare", "Sex", "Embarked"]]

In [ ]:
# Inspect the selected columns: Survived should no longer be present.
X.head(2)

### Reserve the Test Set

`test_size=0.2` reserves about 20% of passengers for the **final evaluation**; the other 80% are used for training and cross-validation. `random_state=42` makes this split repeatable. `stratify=y` approximately preserves the survival proportions in both sets.

Training folds will guide scaling and hyperparameter choices. Do not use held-out test accuracy to choose a configuration, then report that same score as an independent evaluation.

In [ ]:
# Split features and labels together so each passenger keeps the correct label.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

---

## Feature Scaling

Often the input features of your model have different units which means that the variables also have different scales. While some model types (e.g. tree-based models like decision tree or random forest) generally do not require scaling of numerical input variables, many machine learning algorithms including for example algorithms using distance measures (e.g. KNN, SVM) can perform better when numerical features are scaled appropriately; improvement is not guaranteed.

Two common techniques are **min-max scaling** (called normalization in the figure) and **standardization**. Scikit-learn's `Normalizer` instead rescales each row to unit norm; it is not the same as min-max scaling.

See [this guide to scaling transforms](https://machinelearningmastery.com/standardscaler-and-minmaxscaler-transforms-in-python/) for further info.

![Original data compared with min-max-scaled and standardized data](assets/normalization_vs_standardization.png)

The original dataset in the figure contains observations with different value ranges across the two features. The figure demonstrates two common scaling techniques:

- **Min-max scaling** (called normalization in the figure): Subtract the training minimum, then divide by the training range. For a nonconstant feature, the training minimum becomes 0, the maximum becomes 1, and values in between are mapped proportionally. New values can fall outside [0, 1].
- **Standardization**: Subtract the training mean, then divide by the training standard deviation. Nonconstant training features become centered on 0 with population standard deviation 1. This makes their scales comparable, but does not make their distributions Gaussian.

Both methods preserve each feature's ordering and distribution shape while changing its numerical scale. Different feature-wise rescaling changes distances between observations, which is exactly why KNN predictions can change. Constant features are handled separately; they cannot have variance 1.

### How the Preprocessing Pipeline Works

Models need numerical, non-missing inputs. A `ColumnTransformer` sends the two kinds of columns through different steps, then combines their outputs:

| Input columns | Steps in order | Why? |
| --- | --- | --- |
| Numerical | Fill missing values with the training median (middle value), then optionally scale | Keep rows with missing ages and make units comparable |
| Categorical | Fill missing values with the training mode (most frequent category), then one-hot encode | Replace text with indicator columns without imposing a numerical order |

One-hot encoding creates a 0/1 column for each learned category: a passenger recorded as female has `Sex_female=1` and `Sex_male=0` when both categories are learned. `handle_unknown="ignore"` represents an unseen category with all zeros in its category block; it does not learn a new category at prediction time.

A `Pipeline` links preprocessing to the classifier. During cross-validation, each copy learns medians, categories and scaling statistics **only from its training fold**, then applies them unchanged to the validation fold. `"passthrough"` skips scaling, not imputation or encoding. Creating the objects below does not fit anything yet.

[Reference: scikit-learn heterogeneous preprocessing](https://scikit-learn.org/stable/auto_examples/compose/plot_column_transformer_mixed_types.html).

In [ ]:
# Route numeric and categorical inputs to different preprocessing steps.
col_scale = ["Age", "SibSp", "Parch", "Fare"]
col_encode = ["Sex", "Embarked"]


def make_preprocessor(scaler="passthrough"):
    """Build preprocessing steps; the model pipeline fits them within each fold."""
    # Impute before scaling; passthrough provides the unscaled comparison.
    numeric = Pipeline(
        [
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", scaler),
        ]
    )
    # Fill missing categories before encoding; learn both steps on training folds.
    categorical = Pipeline(
        [
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ]
    )
    # Combine the transformed numeric and categorical columns into one model input.
    return ColumnTransformer(
        [
            ("numeric", numeric, col_scale),
            ("categorical", categorical, col_encode),
        ]
    )

### Standardization

Standardization puts numerical features on a comparable scale. First subtract the training mean to center the values around zero, then divide by the training standard deviation to express their distance from the mean in standard-deviation units. For a nonconstant feature:

$$x_{scaled} = \frac{x-\mu_{train}}{\sigma_{train}}$$

Where:

- $x$ is the individual value to transform.
- $\mu_{train}$ (mu) is the **mean** of the training feature values.
- $\sigma_{train}$ (sigma) is their **standard deviation**, which measures how spread out the values are around the mean.

For $m$ observed training values $x_1, \ldots, x_m$, these statistics are calculated as:

$$\mu_{train} = \frac{1}{m}\sum_{i=1}^{m}x_i$$

$$\sigma_{train} = \sqrt{\frac{1}{m}\sum_{i=1}^{m}(x_i-\mu_{train})^2}$$

Here, $m$ is the number of observed training values, $i$ indexes them, and $\sum$ means to add them together. `StandardScaler` uses this population standard deviation (division by $m$, not $m-1$).

**Example:** If the training mean is 50 and the standard deviation is 10, a value of 70 becomes $(70-50)/10=2$: two standard deviations above the training mean. A value of 50 becomes 0, and a value of 40 becomes -1.

**After transformation:** Nonconstant training features have mean 0 and standard deviation 1. Constant columns become 0 after centering. Standardization preserves distribution shape; it does not make the data Gaussian.

Fit the scaler on training data only; inside cross-validation, fit it separately in each training fold. Reuse those statistics on validation or test data, whose transformed mean and standard deviation need not be 0 and 1.

[`StandardScaler`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html) is sensitive to outliers.

In [ ]:
# Create a reusable transformer; fitting happens later inside cross-validation.
scaling_transformation = make_preprocessor(StandardScaler())

### Min-Max Scaling

Min-max scaling (sometimes called normalization) rescales a feature to a chosen range, by default 0 to 1. Subtract the training minimum, then divide by the training range. For a nonconstant feature:

$$x_{scaled} = \frac{x-x_{min,train}}{x_{max,train}-x_{min,train}}$$

Where:

- $x$ is the value to transform.
- $x_{min,train}$ is the smallest observed training value for this feature.
- $x_{max,train}$ is the largest observed training value. Their difference is the training range.

**Example:** With a training minimum of 10 and maximum of 30, a value of 20 becomes $(20-10)/(30-10)=0.5$. A new value of 40 becomes 1.5: do not recompute the minimum and maximum on test data to force it into [0, 1].

[`MinMaxScaler`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.MinMaxScaler.html) learns these bounds from training data and reuses them on new observations. Nonconstant training features map to [0, 1]; constant training features map to 0 with the default settings. It does not remove outliers, and extreme training values can compress most other values into a small part of the range.

### Exercise 1: Practice Round

Create the same preprocessing pipeline with min-max scaling instead of standardization. Do not fit it on all data before cross-validation.

<details>
<summary>Show answer</summary>

```python
from sklearn.preprocessing import MinMaxScaler

normalization_transformation = make_preprocessor(MinMaxScaler())
```

`MinMaxScaler` rescales each numerical column using the minimum and maximum learned from training data, mapping its training range to 0-1. Imputation and categorical encoding are unchanged. The pipeline will fit every learned step separately within each training fold.
</details>

In [ ]:
# Scaling with MinMaxScaler

# Try to scale your data with the MinMaxScaler() from sklearn.
# It follows the same syntax as the StandardScaler.
# Don't forget: you have to import the scaler at the top of your notebook.

### Bonus Question

Do standardized or min-max-scaled test values necessarily have the same properties as transformed training values?

For nonconstant training features, that means a mean of 0 and a standard deviation of 1 after standardization, or values between 0 and 1 after min-max scaling.

<details>
<summary>Show answer</summary>

Not necessarily. The scaler learns its parameters from **training data only**, then applies those same fixed parameters to test data:

- **Standardization**: It reuses the training mean and standard deviation. Transformed test features need not have mean 0 or standard deviation 1, or even be close if the distribution has shifted.
- **Min-max scaling**: It reuses the training minimum and maximum. Test values below that minimum or above that maximum fall outside [0, 1], potentially by a large amount.
- **Constant features**: Even in training data, a constant feature cannot have variance 1; with the default centering it becomes 0.

This is expected: reuse the fitted scaler so the model receives the same representation. Fitting a second scaler on the test set would use held-out information and change that representation.
</details>

---

## Predictive Modelling

We will evaluate model performance in a quick and more reliable way using sklearn's [cross_val_score()](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.cross_val_score.html), which implements K-fold cross-validation. When we train a model on a single train-test split we only have one experiment. Can we really trust one experiment?

Think of [K-fold cross-validation](https://en.wikipedia.org/wiki/Cross-validation_(statistics)) as running K experiments and taking the average error. It is still not perfect, but it is more trustworthy than a single split that might turn out unusually good or bad by chance.

This raises the question of which value to use for K. Common values are between 5 and 10, and you also need to account for technical limits: dataset size, compute power, available memory, and time. Cross-validation takes a while on large datasets.

![Five-fold cross-validation with a different validation fold in each round](assets/cross_validation.png)

For a fair comparison, every candidate below uses the same shuffled stratified five-fold split (`cv`) and accuracy metric. Stratification approximately preserves class proportions; it does not prevent leakage between related passengers such as family members.

The training set is divided into **5 folds** with the current `N_SPLITS` setting. In each round:

- One fold is held out for **validation**.
- The other four folds fit a fresh copy of the preprocessing and model.
- That copy predicts the validation fold and produces one accuracy score.

Each fold is used for validation once, so `cross_val_score` returns five scores. Their mean summarizes performance; their standard deviation describes variability, **not a confidence interval**, because the training folds overlap. These validation folds come only from `X_train`; `X_test` is still held aside.

**After changing folds or the search grid, restart the kernel and rerun all cells** (including exercises you complete). Saved outputs may reflect earlier settings; compare results from the same run.

### Baseline: Always Predict the Majority Class

Before trying KNN or SGD, establish a simple reference. `DummyClassifier(strategy="most_frequent")` ignores the features and always predicts the most common label in each training fold. Evaluate it using the same folds and accuracy metric as the other models; do not calculate the majority class from the full dataset or the test set.

A model should improve on this reference to justify its complexity. Beating it still does not establish real-world usefulness. [Reference: DummyClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.dummy.DummyClassifier.html).

In [ ]:
# Learn the majority label independently inside each training fold.
baseline_scores = cross_val_score(
    DummyClassifier(strategy="most_frequent"),
    X_train,
    y_train,
    scoring="accuracy",
    cv=cv,
    n_jobs=-1,
)
# Retain fold scores so later examples can be compared without rerunning CV.
cv_scores = {"Majority-class baseline": baseline_scores}
print(f"Baseline mean CV accuracy: {baseline_scores.mean():.3f}")

### KNN Classifier: Unscaled Data

K-nearest neighbors (KNN) finds the most similar training passengers using a distance measure, then combines their labels to predict survival. Large numerical ranges can dominate that distance: a fare difference of 100 can outweigh an age difference of 10. Scaling changes how the numerical inputs contribute.

Here, "unscaled" still includes missing-value handling and categorical encoding. `cross_val_score` fits a new pipeline for each fold and returns **validation accuracies**, not the reserved test score.

In [ ]:
# Evaluate the complete preprocessing/model pipeline within each CV fold.
knn_classifier = Pipeline(
    [
        ("preprocessing", make_preprocessor()),
        ("knn", KNeighborsClassifier()),
    ]
)
scores = cross_val_score(
    knn_classifier, X_train, y_train, scoring="accuracy", cv=cv, n_jobs=-1
)
cv_scores["KNN - unscaled"] = scores
print(f"Mean accuracy (unscaled): {scores.mean():.3f}")
print(f"Fold standard deviation (unscaled): {scores.std():.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
# One bar per validation fold; adapt automatically if N_SPLITS changes.
sns.barplot(x=range(1, len(scores) + 1), y=scores, ax=ax)
ax.axhline(scores.mean(), color="darkorange", label="Mean accuracy")
ax.set(
    title="KNN: unscaled inputs",
    xlabel="Validation fold",
    ylabel="Accuracy",
    ylim=(0, 1),
)
ax.legend()
fig.tight_layout()

Image description:
The plot titled "KNN: unscaled inputs" shows one accuracy bar per validation fold (five with the current configuration). The horizontal line is their mean. Compare bar heights to inspect variability and read the current output rather than expecting fixed values across runs or package versions.
End of image description

### KNN Classifier: Scaled Data

In [ ]:
# Fit and evaluate using cross-validation; the held-out test is not used.
# Use the same folds and preprocessing choices; change only numeric scaling.
knn_classifier_scaled = Pipeline(
    [
        ("scaling_trans", scaling_transformation),
        ("knn", KNeighborsClassifier()),
    ]
)
scores_scaled = cross_val_score(
    knn_classifier_scaled, X_train, y_train, scoring="accuracy", cv=cv, n_jobs=-1
)
cv_scores["KNN - standardized"] = scores_scaled
print(f"Mean accuracy (scaled): {scores_scaled.mean():.3f}")
print(f"Fold standard deviation (scaled): {scores_scaled.std():.3f}")
print(
    f"Accuracy difference: {100 * (scores_scaled.mean() - scores.mean()):+.1f} percentage points"
)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
# One bar per validation fold; adapt automatically if N_SPLITS changes.
sns.barplot(x=range(1, len(scores_scaled) + 1), y=scores_scaled, ax=ax)
ax.axhline(scores_scaled.mean(), color="darkorange", label="Mean accuracy")
ax.set(
    title="KNN: standardized inputs",
    xlabel="Validation fold",
    ylabel="Accuracy",
    ylim=(0, 1),
)
ax.legend()
fig.tight_layout()

Compare the scaled and unscaled mean accuracy and the variability across folds. Scaling may improve performance, but neither the size nor the direction of the change is guaranteed. A change from 0.70 to 0.75 is 5 percentage points, not a 5% relative increase. To show the effect of scaling more clearly, we will look at another classifier that is even more sensitive to it.

### SGD Classifier: Unscaled Data

The [SGD classifier](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.SGDClassifier.html) (stochastic gradient descent classifier) implements a variety of classifiers. The model it fits is controlled by the `loss` parameter: by default it fits a linear support vector machine (SVM). We use it here to emphasise the importance of scaling, and set `loss="log_loss"` so the model becomes a logistic regression trained with stochastic gradient descent.

`max_iter=10000` is an upper limit on passes over the training data, not a promise to run all of them. `random_state=RSEED` makes its stochastic training repeatable. The pipeline changes numerical scaling in the next example while keeping the model settings and validation folds the same.

In [ ]:
# Unscaled comparison: imputation and encoding still happen inside each fold.
sgd_classifier = Pipeline(
    [
        ("preprocessing", make_preprocessor()),
        # log_loss trains a linear probabilistic classifier using SGD.
        ("sgd", SGDClassifier(random_state=RSEED, loss="log_loss", max_iter=10000)),
    ]
)
scores = cross_val_score(
    sgd_classifier, X_train, y_train, scoring="accuracy", cv=cv, n_jobs=-1
)
cv_scores["SGD - unscaled"] = scores
print(f"Mean accuracy (unscaled): {scores.mean():.3f}")
print(f"Fold standard deviation (unscaled): {scores.std():.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
# One bar per validation fold; adapt automatically if N_SPLITS changes.
sns.barplot(x=range(1, len(scores) + 1), y=scores, ax=ax)
ax.axhline(scores.mean(), color="darkorange", label="Mean accuracy")
ax.set(
    title="SGD: unscaled inputs",
    xlabel="Validation fold",
    ylabel="Accuracy",
    ylim=(0, 1),
)
ax.legend()
fig.tight_layout()

Inspect how the validation accuracy varies across the different folds. Larger values mean better performance; use the current results rather than assuming a fixed range.

### SGD Classifier: Scaled Data

In [ ]:
# Standardization is fitted on each training fold, not on validation rows.
sgd_classifier_scaled = Pipeline(
    [
        ("scaling_trans", make_preprocessor(StandardScaler())),
        ("sgd", SGDClassifier(random_state=RSEED, loss="log_loss", max_iter=10000)),
    ]
)
scores_scaled = cross_val_score(
    sgd_classifier_scaled, X_train, y_train, scoring="accuracy", cv=cv, n_jobs=-1
)
cv_scores["SGD - standardized"] = scores_scaled
print(f"Mean accuracy (scaled): {scores_scaled.mean():.3f}")
print(f"Fold standard deviation (scaled): {scores_scaled.std():.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
# One bar per validation fold; adapt automatically if N_SPLITS changes.
sns.barplot(x=range(1, len(scores_scaled) + 1), y=scores_scaled, ax=ax)
ax.axhline(scores_scaled.mean(), color="darkorange", label="Mean accuracy")
ax.set(
    title="SGD: standardized inputs",
    xlabel="Validation fold",
    ylabel="Accuracy",
    ylim=(0, 1),
)
ax.legend()
fig.tight_layout()

Compare both the mean validation accuracy and its variability across folds before and after scaling. Scaling can help SGD optimization, but improved accuracy and reduced variability are not guaranteed.

### Exercise 2: Practice Round

Try the KNN classifier with the min-max-scaled data.

<details>
<summary>Show answer</summary>

Reuse the normalization transformer from Exercise 1 inside a `Pipeline`, just as we did for the standardized KNN model, then evaluate it with cross-validation:

```python
knn_normalized = Pipeline(
    [("scaling_trans", normalization_transformation), ("knn", KNeighborsClassifier())]
)
scores_norm = cross_val_score(
    knn_normalized, X_train, y_train, scoring="accuracy", cv=cv, n_jobs=-1
)
print("Score (normalized):", round(scores_norm.mean(), 2))
cv_scores["KNN - min-max"] = scores_norm
```

Both scalers change distances, but they need not give similar results. Compare them on the same validation folds; neither is universally better.
</details>

In [ ]:
# Train model using the MinMaxScaler data you scaled in ex1.

# It follows the same syntax as the StandardScaler.

### Scaling Comparison at a Glance

Collect the previous fold scores in one table. The gain is the difference from the baseline in **percentage points**, not a relative percentage. If you completed Exercise 2 using the answer above, the min-max result also appears. No extra models are trained here.

Which model improves on the baseline most? How does scaling affect KNN versus SGD? Read the fold standard deviation as variability, not as a confidence interval or proof of a statistically significant difference. This table compares untuned models; the KNN searches below are a separate demonstration.

In [ ]:
# Summarize stored fold scores; this cell does not refit any model.
cv_comparison = pd.DataFrame(
    {
        name: {
            "mean_cv_accuracy": fold_scores.mean(),
            "fold_standard_deviation": fold_scores.std(),
            "gain_vs_baseline_pp": 100 * (fold_scores.mean() - baseline_scores.mean()),
        }
        for name, fold_scores in cv_scores.items()
    }
).T
cv_comparison.round(3)

---

## Hyperparameter Tuning

Hyperparameters are settings we choose before fitting, such as KNN's number of neighbors; learned parameters are obtained from training data. Most models have many hyperparameters, and the best values differ from one dataset to another. The same is true for the regularization hyperparameters we saw earlier, which are usually chosen by trial and error. So how do we find the parameter values that work best for our data?

### GridSearchCV

[Grid search](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html) is a tuning technique that searches for the best hyperparameter values. It performs an exhaustive search over a predefined parameter space using cross-validation (hence the **CV** suffix), evaluating every possible combination in the search space to find and return the best one.

This becomes time-consuming when there are many hyperparameters and the search space is large. In the illustrative figure below, with k=5 and 2 parameters taking 2 and 3 values respectively (6 combinations), GridSearchCV runs 30 modelling steps just to find the best values for those two parameters.

![Grid search evaluates six parameter combinations with five-fold cross-validation](assets/grid_search_cv.png)

The process evaluates multiple combinations of hyperparameters to identify the model configuration that performs best.

In the figure (not the code below), two hyperparameters are considered:

- parameter **a** with possible values **1, 2, and 3**
- parameter **b** with possible values **1 and 2**

This produces **6 possible hyperparameter combinations**.

For each combination, **5-fold cross-validation** is performed:

- the training data is divided into 5 folds
- the model is trained and validated 5 times
- the performance scores are averaged to obtain a cross-validation score for that parameter combination

This process is repeated for all 6 combinations.

In total, **30 cross-validation fits are performed (6 × 5)**. With the default `refit=True`, the best configuration is then fitted once more on all the training data supplied to the search.

The parameter combination with the best average validation performance is selected as the final model configuration.

#### Configure and Observe the Search

Pipeline hyperparameter names use `step__parameter`: `knn__n_neighbors` changes `n_neighbors` in the step named `knn`.

| KNN setting | Meaning |
| --- | --- |
| `n_neighbors` | How many nearby training passengers vote |
| `weights="uniform"` / `"distance"` | Equal votes / greater influence for closer neighbors |
| `p=1` / `2` / `3` | Manhattan / Euclidean / another Minkowski distance |

The grid below has **5 x 2 x 3 = 30 combinations**. With five folds, grid search performs **150 cross-validation fits**, then one additional fit of the winner on all of `X_train` (`refit=True` by default). For each fold it refits the **complete pipeline**, not just KNN.

`verbose=3` makes this work visible: an initial fit-count summary, followed by candidate/fold information, scores and timing. These scores are **validation scores**, not final test accuracy. Set `verbose=1` for a shorter summary. With `n_jobs=-1`, available CPU cores work in parallel, so detailed messages may interleave or appear in the editor's kernel output.

[Reference: GridSearchCV verbosity and refitting](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html).

In [ ]:
# Inspect nested settings; knn__ names address the classifier inside the pipeline.
knn_classifier_scaled.get_params()

In [ ]:
# List the values to try; grid search evaluates their Cartesian product.
param_grid = {
    "knn__n_neighbors": [2, 3, 4, 5, 10],  # number of neighbors
    "knn__weights": ["uniform", "distance"],  # equal or distance-weighted votes
    "knn__p": [1, 2, 3],  # Minkowski distance: 1 = Manhattan, 2 = Euclidean
}

# Use the same validation folds as before; show detailed progress with verbose.
gs = GridSearchCV(
    knn_classifier_scaled, param_grid, scoring="accuracy", cv=cv, verbose=3, n_jobs=-1
)

# Fit only on training data: search, select by mean CV accuracy, then refit.
# Measure elapsed seconds for this run, not a general speed benchmark.
start = timer()
gs.fit(X_train, y_train)
end = timer()
gs_time = end - start

In [ ]:
gs_time

In [ ]:
# Mean validation accuracy of the winning candidate, not test accuracy.
print("Best CV accuracy:", round(gs.best_score_, 3))

# Chosen hyperparameter values; the learned model is in best_estimator_.
print("Best parameters:", gs.best_params_)

In [ ]:
# Keep the reporting logic together; call it only for the final selected model.
def print_pretty_summary(name, best_model, y_test, y_pred_test):
    print(name)
    print("=======================")
    # The last pipeline step contains the fitted KNN classifier.
    model = best_model.steps[-1][1]
    print(f"n_neighbors: {model.n_neighbors}")
    print(f"weights: {model.weights}")
    print(f"p: {model.p}")
    print(f"algorithm: {model.algorithm}")

    accuracy = accuracy_score(y_test, y_pred_test)
    print(f"Final test accuracy: {accuracy:.3f}")
    return accuracy

In [ ]:
# Already refitted on all X_train, including its imputer, encoder and scaler.
knn_best = gs.best_estimator_
knn_best

In [ ]:
# Keep the test set untouched until both searches have finished.
# Compare cross-validation scores, not test scores, to select the model.
print(f"Grid search best CV accuracy: {gs.best_score_:.3f}")

### RandomizedSearchCV

As an alternative to grid search we can use sklearn's [RandomizedSearchCV()](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.RandomizedSearchCV.html). Random search does not try every possible combination in the search space; instead it randomly picks and evaluates a set number of parameter combinations.

Reuse **the same 30-combination space and the same folds** so the comparison changes only the search strategy. `n_iter=10` samples ten distinct combinations from these lists: **50 CV fits**, then one refit. `random_state=RSEED` makes that sample repeatable. The logs use the same `verbose=3` setting as grid search.

In [ ]:
# Reuse param_grid from grid search; sample fewer candidates from the same space.
rs = RandomizedSearchCV(
    knn_classifier_scaled,
    param_grid,
    scoring="accuracy",
    cv=cv,
    n_iter=10,
    verbose=3,
    n_jobs=-1,
    random_state=RSEED,
)

# Search only on X_train, with fold-local preprocessing and a final training refit.
start = timer()
rs.fit(X_train, y_train)
end = timer()
rgs_time = end - start

In [ ]:
# Compare this mean validation accuracy with grid search, not with test accuracy.
print("Best CV accuracy:", round(rs.best_score_, 3))

# Best parameters
print("Best parameters:", rs.best_params_)

### Choose Before Testing

`best_score_` is the winning mean CV accuracy; `best_params_` contains its settings; `best_estimator_` is the complete fitted pipeline. Because the search optimizes CV accuracy, the best CV score may be optimistic; the reserved test set gives a separate final check.

The table compares the **two tuned KNN searches**. The earlier KNN/SGD examples illustrate scaling and are not candidates in this final selection. Choose using CV accuracy, then call `predict(X_test)` once with the selected pipeline. Prediction reuses learned preprocessing; it does not fit the model again.

In [ ]:
# Compare accuracy, number of tried configurations and elapsed time.
searches = {"Grid search": gs, "Randomized search": rs}
comparison = pd.DataFrame(
    {
        "best_cv_accuracy": [gs.best_score_, rs.best_score_],
        "candidates": [len(gs.cv_results_["params"]), len(rs.cv_results_["params"])],
        "elapsed_seconds": [gs_time, rgs_time],
    },
    index=list(searches),
)
comparison

In [ ]:
# Select before seeing test predictions; ties keep the first dictionary entry.
selected_name = max(searches, key=lambda name: searches[name].best_score_)
selected_model = searches[selected_name].best_estimator_
# First and final test evaluation: apply the fitted pipeline without retraining.
y_pred_test = selected_model.predict(X_test)
print(f"Selected by CV accuracy: {selected_name}")
final_test_accuracy = print_pretty_summary(
    selected_name, selected_model, y_test, y_pred_test
)

## Interpretation

The final evaluation uses the model already refitted on all training data by the chosen search. In an exact CV tie, the code selects grid search because it appears first in the predefined dictionary.

Randomized search evaluates 10 of the 30 listed candidates here; grid search evaluates all 30. Runtime depends on hardware and caching, so one timing is not a benchmark. Do not use the test score to go back and tune again. Accuracy is suitable for this introductory comparison but does not describe every kind of classification error.

## Export for Notebook 3

After selection and final evaluation, save the fitted pipeline and a small metadata file. This reuses `selected_model` exactly as evaluated; there is no new training or tuning. Notebook 3 loads these files in a fresh kernel to predict on invented requests.

The files stay in the Git-ignored `artifacts/` directory. Save the complete pipeline, including preprocessing. [Reference: scikit-learn model persistence](https://scikit-learn.org/stable/model_persistence.html).

In [ ]:
import json
from importlib.metadata import version
from pathlib import Path

import joblib

artifact_dir = Path("artifacts")
artifact_dir.mkdir(exist_ok=True)
# Read the learned categories from the selected pipeline; do not fit again.
fitted_preprocessor = selected_model.steps[0][1]
fitted_encoder = fitted_preprocessor.named_transformers_["categorical"].named_steps[
    "encoder"
]
# Keep raw input names and training ranges so notebook 3 can validate new requests.
model_info = {
    "selected_search": selected_name,
    "input_features": list(X_train.columns),
    "numeric_features": col_scale,
    "categorical_features": col_encode,
    "known_categories": {
        column: categories.tolist()
        for column, categories in zip(
            col_encode, fitted_encoder.categories_, strict=True
        )
    },
    "numeric_training_ranges": X_train[col_scale].agg(["min", "max"]).to_dict(),
    "package_versions": {
        package: version(package)
        for package in ["scikit-learn", "numpy", "pandas", "joblib"]
    },
    "evaluation": {
        "cv_accuracy": float(searches[selected_name].best_score_),
        "test_accuracy": float(final_test_accuracy),
    },
}
# Persist the fitted model; the metadata describes its inputs and existing evaluation.
joblib.dump(selected_model, artifact_dir / "titanic_pipeline.joblib")
(artifact_dir / "titanic_model_info.json").write_text(
    json.dumps(model_info, indent=2) + "\n"
)
print("Saved the fitted pipeline and metadata for notebook 3 in artifacts/.")